# Combine movie metadata

Merge TMDB movies and credits, then join with Netflix `movie_titles` on normalized title + year. Save the result to `data/processed/`.

## 1. Load raw files

Read TMDB movies, TMDB credits, and Netflix titles. Netflix titles have no header and may contain commas, so they are parsed line by line.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(RAW / "tmdb_5000_movies.csv")
credits = pd.read_csv(RAW / "tmdb_5000_credits.csv")

rows = []
with open(RAW / "movie_titles.csv", encoding="latin-1") as f:
    for line in f:
        movie_id, year, title = line.rstrip("\n").split(",", 2)
        rows.append(
            {
                "netflix_movie_id": int(movie_id),
                "year": None if year == "NULL" else int(year),
                "netflix_title": title,
            }
        )
netflix = pd.DataFrame(rows)

print("movies:", movies.shape)
print("credits:", credits.shape)
print("netflix:", netflix.shape)
display(movies.head(2))
display(credits.head(2))
display(netflix.head(2))

movies: (4803, 20)
credits: (4803, 4)
netflix: (17770, 3)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


,netflix_movie_id,year,netflix_title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review


## 2. Merge TMDB movies and credits

Join on TMDB id (`movies.id` = `credits.movie_id`). Keep cast and crew from credits.

In [2]:
tmdb = movies.merge(
    credits.drop(columns=["title"]),
    left_on="id",
    right_on="movie_id",
    how="inner",
)
tmdb = tmdb.drop(columns=["movie_id"]).rename(columns={"id": "tmdb_id"})

print("tmdb merged:", tmdb.shape)
tmdb.head(2)

tmdb merged: (4803, 22)


,budget,genres,homepage,tmdb_id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## 3. Prepare Netflix titles for matching

Normalize titles (lowercase, strip, collapse whitespace) and derive year from TMDB `release_date`.

In [3]:
def normalize_title(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

netflix = netflix.copy()
netflix["title_norm"] = normalize_title(netflix["netflix_title"])

tmdb = tmdb.copy()
tmdb["year"] = pd.to_datetime(tmdb["release_date"], errors="coerce").dt.year
tmdb["title_norm"] = normalize_title(tmdb["title"])

print("Netflix rows with year:", netflix["year"].notna().sum())
print("TMDB rows with year:", tmdb["year"].notna().sum())
netflix[["netflix_movie_id", "year", "netflix_title", "title_norm"]].head(3)

Netflix rows with year: 17763
TMDB rows with year: 4802


,netflix_movie_id,year,netflix_title,title_norm
0,1,2003.0,Dinosaur Planet,dinosaur planet
1,2,2004.0,Isle of Man TT 2004 Review,isle of man tt 2004 review
2,3,1997.0,Character,character


## 4. Join Netflix with TMDB

Inner join on normalized title + year so each row has a Netflix id and TMDB metadata.

In [4]:
joined = netflix.merge(
    tmdb,
    on=["title_norm", "year"],
    how="inner",
    suffixes=("_netflix", "_tmdb"),
)

dup_keys = (
    joined.groupby(["title_norm", "year"], dropna=False)
    .size()
    .reset_index(name="n")
)
n_dup_keys = (dup_keys["n"] > 1).sum()

print("Joined rows:", len(joined))
print("Unique Netflix ids:", joined["netflix_movie_id"].nunique())
print("Unique TMDB ids:", joined["tmdb_id"].nunique())
print("Unmatched Netflix titles:", len(netflix) - joined["netflix_movie_id"].nunique())
print("Unmatched TMDB titles:", len(tmdb) - joined["tmdb_id"].nunique())
print("Duplicate title+year keys in join:", n_dup_keys)

joined = joined.rename(columns={"title": "tmdb_title"})
cols = [
    "netflix_movie_id",
    "tmdb_id",
    "netflix_title",
    "tmdb_title",
    "year",
] + [
    c
    for c in joined.columns
    if c
    not in {
        "netflix_movie_id",
        "tmdb_id",
        "netflix_title",
        "tmdb_title",
        "year",
        "title_norm",
    }
]
movies_metadata = joined[cols].copy()
movies_metadata.head(3)

Joined rows: 1806
Unique Netflix ids: 1806
Unique TMDB ids: 1806
Unmatched Netflix titles: 15964
Unmatched TMDB titles: 2997
Duplicate title+year keys in join: 0


,netflix_movie_id,tmdb_id,netflix_title,tmdb_title,year,budget,genres,homepage,keywords,original_language,...,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast,crew
0,3,17139,Character,Character,1997.0,4500000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10769, ""n...",NaN,"[{""id"": 3683, ""name"": ""law and ethics""}]",nl,...,1997-04-17,0,122.0,"[{""iso_639_1"": ""nl"", ""name"": ""Nederlands""}, {""...",Released,NaN,7.7,25,"[{""cast_id"": 4, ""character"": ""Dreverhaven"", ""c...","[{""credit_id"": ""52fe47099251416c7508bca3"", ""de..."
1,30,6964,Something's Gotta Give,Something's Gotta Give,2003.0,80000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",NaN,"[{""id"": 965, ""name"": ""age difference""}, {""id"":...",en,...,2003-12-12,266728738,128.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,Schmucks are people too.,6.3,410,"[{""cast_id"": 1, ""character"": ""Harry Sanborn"", ...","[{""credit_id"": ""548200699251416e7800666d"", ""de..."
2,55,11863,Jade,Jade,1995.0,50000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 53, ""nam...",NaN,"[{""id"": 550, ""name"": ""callgirl""}, {""id"": 582, ...",en,...,1995-10-13,9851610,95.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Some fantasies go too far.,5.2,44,"[{""cast_id"": 1, ""character"": ""David Corelli"", ...","[{""credit_id"": ""52fe44959251416c75039f45"", ""de..."


## 5. Save processed CSV

Write the joined dataframe to `data/processed/movies_metadata.csv`.

In [5]:
out_path = PROCESSED / "movies_metadata.csv"
movies_metadata.to_csv(out_path, index=False)
print(f"Saved {len(movies_metadata):,} rows x {movies_metadata.shape[1]} columns to {out_path.resolve()}")

Saved 1,806 rows x 25 columns to C:\Users\Revital\Documents\לימודים\Project\data\processed\movies_metadata.csv
